# Application Insights — Gen AI Monitoring

This notebook queries an Azure Monitor Log Analytics workspace to surface observability data for Gen AI workloads instrumented through Application Insights. It covers three views:

* **Content traces** — recent rows from `AppGenAIContent` showing agent interactions, model inputs/outputs, tool calls, and evaluation explanations.
* **Tool-call performance** — daily volume, failure rate, and latency percentiles (avg and p95) for Gen AI tool-call spans in `AppDependencies`, grouped by agent, operation, and tool.
* **Token usage** — daily input, output, and total token consumption from `AppDependencies`, broken down by model, provider, and operation type.

Authentication uses a service principal whose credentials are stored in Azure Key Vault and accessed via a Key Vault-backed Databricks secret scope.

## Configure access

1. In **Application Insights → Overview**, find the linked Log Analytics workspace. Copy that workspace's **Workspace ID** — not its Azure resource ID or the Application Insights instrumentation key.
2. Give a Microsoft Entra service principal the **Log Analytics Reader** role on that workspace.
3. Store the service principal's **tenant ID**, **client ID**, and **client secret** as secrets in an [Azure Key Vault](https://learn.microsoft.com/azure/key-vault/general/overview). Then create a [Key Vault-backed secret scope](https://learn.microsoft.com/azure/databricks/security/secrets/secret-scopes#create-an-azure-key-vault-backed-secret-scope) in Databricks so the notebook can retrieve them with `dbutils.secrets.get(scope="<key-vault-scope>", key="<secret-name>")`.

> **Note:** This notebook uses explicit service-principal authentication via Azure Key Vault secrets. Your Databricks login does not automatically grant access to Azure Monitor.

## Install the SDK

In [0]:
%pip install azure-monitor-query azure-identity

In [0]:
dbutils.library.restartPython()

## Query Application Insights

### Query GenAI content traces
Retrieves the 100 most recent rows from the `AppGenAIContent` table, which captures Gen AI span data — including agent names, model identifiers, input/output messages, tool calls, and evaluation explanations.

In [0]:
import pandas as pd
from azure.identity import ClientSecretCredential
from azure.monitor.query import LogsQueryClient, LogsQueryStatus

key_vault_name = "DatabricksKV-JackBender"

# Authenticate using service principal credentials from Key Vault
credential = ClientSecretCredential(
    tenant_id=dbutils.secrets.get(scope=key_vault_name, key="SPTenantID"),
    client_id=dbutils.secrets.get(scope=key_vault_name, key="SPDatabricksAppID"),
    client_secret=dbutils.secrets.get(scope=key_vault_name, key="SPDatabricks"),
)

workspace_id = "e3733134-7f65-42d5-aeab-2c022c5e44a9"

query = """
AppGenAIContent
| top 100 by TimeGenerated desc
"""

with LogsQueryClient(credential) as client:
    result = client.query_workspace(
        workspace_id=workspace_id,
        query=query,
        timespan=timedelta(days=7),
    )

if result.status != LogsQueryStatus.SUCCESS:
    raise RuntimeError(f"Query returned incomplete results: {result.partial_error}")

table = result.tables[0]
logs = pd.DataFrame(table.rows, columns=table.columns)
display(logs)


### Daily tool-call performance by agent
Queries the `AppDependencies` table to identify Gen AI tool-call spans via the `Properties` column, filtering to Microsoft Foundry, Azure AI Agents, or any dependency with a tool-call ID. Summarizes daily tool-call volume, failure rate, average and p95 latency, and sample result codes and targets, grouped by agent, operation, tool name, and role.

In [0]:
from datetime import timedelta

import pandas as pd
from azure.identity import ClientSecretCredential
from azure.monitor.query import LogsQueryClient, LogsQueryStatus

key_vault_name = "DatabricksKV-JackBender"

# Authenticate using service principal credentials from Key Vault
credential = ClientSecretCredential(
    tenant_id=dbutils.secrets.get(scope=key_vault_name, key="SPTenantID"),
    client_id=dbutils.secrets.get(scope=key_vault_name, key="SPDatabricksAppID"),
    client_secret=dbutils.secrets.get(scope=key_vault_name, key="SPDatabricks"),
)

workspace_id = "e3733134-7f65-42d5-aeab-2c022c5e44a9"

query = """
AppDependencies
| extend dims = column_ifexists("Properties", dynamic(null))
| extend genAiOperation = tostring(dims["gen_ai.operation.name"]),
         genAiProvider = tostring(dims["gen_ai.provider.name"]),
         genAiAgentName = tostring(dims["gen_ai.agent.name"]),
         genAiAgentId = tostring(dims["gen_ai.agent.id"]),
         genAiToolName = tostring(dims["gen_ai.tool.name"]),
         genAiToolCallId = tostring(dims["gen_ai.tool.call.id"]),
         spanType = tostring(dims["span_type"]),
         foundryFlag = tostring(dims["microsoft.foundry"])
| where foundryFlag =~ "True"
   or genAiProvider in~ ("microsoft.foundry", "az.ai.agents")
   or genAiOperation =~ "execute_tool"
   or spanType =~ "tool"
   or isnotempty(genAiToolCallId)
| extend agentName = coalesce(genAiAgentName, genAiAgentId),
         itemCountValue = coalesce(tolong(column_ifexists("ItemCount", 1)), 1),
         durationMs = iif(gettype(DurationMs) == "timespan", todouble(DurationMs) * 86400000.0, todouble(DurationMs)),
         resultCodeText = tostring(ResultCode)
| summarize toolCalls = sum(itemCountValue),
            failedToolCalls = sumif(itemCountValue, Success == false),
            failureRatePct = 100.0 * sumif(itemCountValue, Success == false) / sum(itemCountValue),
            avgDurationMs = avg(durationMs),
            p95DurationMs = percentile(durationMs, 95),
            sampleResultCodes = make_set(resultCodeText, 5),
            sampleTargets = make_set(Target, 5)
  by timestamp = bin(TimeGenerated, 1d),
     agentName, genAiOperation, genAiToolName, AppRoleName
| order by timestamp asc, toolCalls desc, failedToolCalls desc, p95DurationMs desc
"""

with LogsQueryClient(credential) as client:
    result = client.query_workspace(
        workspace_id=workspace_id,
        query=query,
        timespan=timedelta(days=31),
    )

if result.status != LogsQueryStatus.SUCCESS:
    raise RuntimeError(f"Query returned incomplete results: {result.partial_error}")

table = result.tables[0]
logs = pd.DataFrame(table.rows, columns=table.columns)
if logs.empty:
    print("No rows returned — the query filters may not match any data in this timespan.")
else:
    display(logs)

### Summarize daily token usage
Queries the `AppDependencies` table to extract Gen AI token metrics from the `Properties` column, then aggregates input, output, and total tokens per day and model. Filters to rows associated with Microsoft Foundry, Azure AI Agents, or any dependency that reports a model name.

In [0]:
from datetime import timedelta

import pandas as pd
from azure.identity import ClientSecretCredential
from azure.monitor.query import LogsQueryClient, LogsQueryStatus

key_vault_name = "DatabricksKV-JackBender"

# Authenticate using service principal credentials from Key Vault
credential = ClientSecretCredential(
    tenant_id=dbutils.secrets.get(scope=key_vault_name, key="SPTenantID"),
    client_id=dbutils.secrets.get(scope=key_vault_name, key="SPDatabricksAppID"),
    client_secret=dbutils.secrets.get(scope=key_vault_name, key="SPDatabricks"),
)

workspace_id = "e3733134-7f65-42d5-aeab-2c022c5e44a9"

query = """
AppDependencies
| extend d = column_ifexists("Properties", dynamic(null))
| extend
    model = coalesce(
        tostring(d["gen_ai.request.model"]),
        tostring(d["gen_ai.response.model"])
    ),
    inputTokens = tolong(d["gen_ai.usage.input_tokens"]),
    outputTokens = tolong(d["gen_ai.usage.output_tokens"]),
    provider = tostring(d["gen_ai.provider.name"]),
    foundry = tostring(d["microsoft.foundry"])
| extend totalTokens = coalesce(inputTokens, 0) + coalesce(outputTokens, 0)
| where foundry =~ "True"
    or provider in~ ("microsoft.foundry", "az.ai.agents")
    or isnotempty(model)
| summarize
    inputTokens = sum(coalesce(inputTokens, 0)),
    outputTokens = sum(coalesce(outputTokens, 0)),
    totalTokens = sum(totalTokens)
    by ts = bin(TimeGenerated, 1d), model
| order by ts asc, totalTokens desc
"""

with LogsQueryClient(credential) as client:
    result = client.query_workspace(
        workspace_id=workspace_id,
        query=query,
        timespan=timedelta(days=31),
    )

if result.status != LogsQueryStatus.SUCCESS:
    raise RuntimeError(f"Query returned incomplete results: {result.partial_error}")

table = result.tables[0]
logs = pd.DataFrame(table.rows, columns=table.columns)
if logs.empty:
    print("No rows returned — the query filters may not match any data in this timespan.")
else:
    display(logs)

### Daily token usage by model and operation
Queries the `AppDependencies` table to extract per-request token counts (input, output, total) from the `Properties` column, then aggregates them by day, model, provider, and operation type. Filters to rows that report a model name or any token usage.

In [0]:
from datetime import timedelta

import pandas as pd
from azure.identity import ClientSecretCredential
from azure.monitor.query import LogsQueryClient, LogsQueryStatus

key_vault_name = "DatabricksKV-JackBender"

# Authenticate using service principal credentials from Key Vault
credential = ClientSecretCredential(
    tenant_id=dbutils.secrets.get(scope=key_vault_name, key="SPTenantID"),
    client_id=dbutils.secrets.get(scope=key_vault_name, key="SPDatabricksAppID"),
    client_secret=dbutils.secrets.get(scope=key_vault_name, key="SPDatabricks"),
)

workspace_id = "e3733134-7f65-42d5-aeab-2c022c5e44a9"

query = """
AppDependencies
| extend dims = column_ifexists("Properties", dynamic(null))
| extend model = coalesce(tostring(dims["gen_ai.request.model"]), tostring(dims["gen_ai.response.model"])),
         provider = tostring(dims["gen_ai.provider.name"]),
         operation = tostring(dims["gen_ai.operation.name"]),
         inputTokens = tolong(dims["gen_ai.usage.input_tokens"]),
         outputTokens = tolong(dims["gen_ai.usage.output_tokens"]),
         totalTokens = tolong(dims["gen_ai.usage.total_tokens"]),
         itemCountValue = coalesce(tolong(column_ifexists("ItemCount", 1)), 1)
| where isnotempty(model) or isnotempty(inputTokens) or isnotempty(outputTokens) or isnotempty(totalTokens)
| summarize
    calls = sum(itemCountValue),
    inputTokens = sum(coalesce(inputTokens, 0)),
    outputTokens = sum(coalesce(outputTokens, 0)),
    totalTokens = sum(coalesce(totalTokens, coalesce(inputTokens, 0) + coalesce(outputTokens, 0)))
    by timestamp = bin(TimeGenerated, 1d), model, provider, operation
| order by timestamp asc, totalTokens desc
"""

with LogsQueryClient(credential) as client:
    result = client.query_workspace(
        workspace_id=workspace_id,
        query=query,
        timespan=timedelta(days=31),
    )

if result.status != LogsQueryStatus.SUCCESS:
    raise RuntimeError(f"Query returned incomplete results: {result.partial_error}")

table = result.tables[0]
logs = pd.DataFrame(table.rows, columns=table.columns)
if logs.empty:
    print("No rows returned — the query filters may not match any data in this timespan.")
else:
    display(logs)
